In [1]:
import torch 
from torchvision import datasets, transforms
from torch.utils.data import random_split

trainfull=datasets.FashionMNIST(root='./data', train=True,download=True,transform=transforms.ToTensor())
test_set=datasets.FashionMNIST(root='./data',train=False,download=True,transform=transforms.ToTensor())

print('Full training set size: ',len(trainfull))
print('test_set size: ', len(test_set))

valsize=5000
trainsize=len(trainfull)-valsize

trainset,valset=random_split(trainfull, [trainsize,valsize],generator=torch.Generator().manual_seed(42))


print("Training set size:", len(trainset))
print("Validation set size:", len(valset))

# Peek at category names
class_names = trainfull.classes
print("\nCategories:", class_names)

Full training set size:  60000
test_set size:  10000
Training set size: 55000
Validation set size: 5000

Categories: ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']


In [4]:
from torchvision import transforms

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((128, 128)),   # lighter than 224x224
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

train_full = datasets.FashionMNIST(root="./data", train=True, download=False, transform=transform)
test_set = datasets.FashionMNIST(root="./data", train=False, download=False, transform=transform)

train_set, val_set = random_split(train_full, [55000, 5000],
                                    generator=torch.Generator().manual_seed(42))

sample_img, sample_label = train_set[0]
print("Image shape:", sample_img.shape)  # should be [3, 128, 128]

Image shape: torch.Size([3, 128, 128])


In [6]:
import torch
import torch.nn as nn
from torchvision import models
from torch.utils.data import DataLoader

torch.device("mps" if torch.backends.mps.is_available() else "cpu")

resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
for param in resnet.parameters():
    param.requires_grad = False
resnet.fc = nn.Identity()
resnet = resnet.to(device)
resnet.eval()

def extract_features(dataset, batch_size=32):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    all_features = []
    all_labels = []
    with torch.no_grad():
        for i, (images, labels) in enumerate(loader):
            images = images.to(device)
            features = resnet(images)
            all_features.append(features.cpu())
            all_labels.append(labels)
            if i % 50 == 0:
                print(f"  batch {i}/{len(loader)}")
    return torch.cat(all_features), torch.cat(all_labels)

print("Extracting training features...")
train_features, train_labels = extract_features(train_set)
print("Extracting validation features...")
val_features, val_labels = extract_features(val_set)
print("Extracting test features...")
test_features, test_labels = extract_features(test_set)

print("Train features shape:", train_features.shape)
print("Val features shape:", val_features.shape)
print("Test features shape:", test_features.shape)

Extracting training features...
  batch 0/1719
  batch 50/1719
  batch 100/1719
  batch 150/1719
  batch 200/1719
  batch 250/1719
  batch 300/1719
  batch 350/1719
  batch 400/1719
  batch 450/1719
  batch 500/1719
  batch 550/1719
  batch 600/1719
  batch 650/1719
  batch 700/1719
  batch 750/1719
  batch 800/1719
  batch 850/1719
  batch 900/1719
  batch 950/1719
  batch 1000/1719
  batch 1050/1719
  batch 1100/1719
  batch 1150/1719
  batch 1200/1719
  batch 1250/1719
  batch 1300/1719
  batch 1350/1719
  batch 1400/1719
  batch 1450/1719
  batch 1500/1719
  batch 1550/1719
  batch 1600/1719
  batch 1650/1719
  batch 1700/1719
Extracting validation features...
  batch 0/157
  batch 50/157
  batch 100/157
  batch 150/157
Extracting test features...
  batch 0/313
  batch 50/313
  batch 100/313
  batch 150/313
  batch 200/313
  batch 250/313
  batch 300/313
Train features shape: torch.Size([55000, 512])
Val features shape: torch.Size([5000, 512])
Test features shape: torch.Size([10000

In [7]:
import torch.optim as optim
from torch.utils.data import TensorDataset

# Wrap our cached features into simple datasets
train_ds = TensorDataset(train_features, train_labels)
val_ds = TensorDataset(val_features, val_labels)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

# The new classifier head: takes 512 numbers in, outputs 10 category scores
head = nn.Linear(512, 10).to(device)

optimizer = optim.Adam(head.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

epochs = 10

for epoch in range(epochs):
    head.train()
    total_loss = 0
    for features, labels in train_loader:
        features, labels = features.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = head(features)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    # Check validation accuracy after each epoch
    head.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for features, labels in val_loader:
            features, labels = features.to(device), labels.to(device)
            outputs = head(features)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_acc = correct / total
    print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss:.3f} - Val Accuracy: {val_acc:.4f}")

Epoch 1/10 - Loss: 409.101 - Val Accuracy: 0.8616
Epoch 2/10 - Loss: 298.482 - Val Accuracy: 0.8696
Epoch 3/10 - Loss: 278.618 - Val Accuracy: 0.8718
Epoch 4/10 - Loss: 269.329 - Val Accuracy: 0.8780
Epoch 5/10 - Loss: 262.747 - Val Accuracy: 0.8738
Epoch 6/10 - Loss: 256.608 - Val Accuracy: 0.8824
Epoch 7/10 - Loss: 252.280 - Val Accuracy: 0.8784
Epoch 8/10 - Loss: 249.741 - Val Accuracy: 0.8770
Epoch 9/10 - Loss: 246.559 - Val Accuracy: 0.8812
Epoch 10/10 - Loss: 244.487 - Val Accuracy: 0.8824


In [9]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import pandas as pd
# Wrap test features into a loader
test_ds = TensorDataset(test_features, test_labels)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

head.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for features, labels in test_loader:
        features, labels = features.to(device), labels.to(device)
        outputs = head(features)
        preds = outputs.argmax(dim=1)
        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())

all_preds = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

# Final test accuracy
test_acc = accuracy_score(all_labels, all_preds)
print("Final Test Accuracy:", test_acc)

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
print("\nConfusion Matrix:")
print(cm_df)

# Per-class precision/recall
print("\nPer-class report:")
print(classification_report(all_labels, all_preds, target_names=class_names))

Final Test Accuracy: 0.886

Confusion Matrix:
             T-shirt/top  Trouser  Pullover  Dress  Coat  Sandal  Shirt  \
T-shirt/top          841        3        13     33     4       1     97   
Trouser                3      973         1     17     2       0      4   
Pullover              18        1       792      6    77       0    103   
Dress                 27       11         6    861    26       1     67   
Coat                   2        1        37     33   839       0     88   
Sandal                 0        0         0      0     0     962      0   
Shirt                117        2        40     31    94       1    707   
Sneaker                0        0         0      0     0      17      0   
Bag                    2        0         0      3     3       3     10   
Ankle boot             0        0         0      0     0       9      1   

             Sneaker  Bag  Ankle boot  
T-shirt/top        0    7           1  
Trouser            0    0           0  
Pullover

In [10]:
import os

os.makedirs("models", exist_ok=True)

# Save just the head's learned weights
torch.save(head.state_dict(), "models/product_classifier.pt")
print("Saved to models/product_classifier.pt")

Saved to models/product_classifier.pt


In [15]:
from PIL import Image

class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

def load_classifier(weights_path="models/product_classifier.pt", device="cpu"):
    # Rebuild the frozen backbone
    backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    backbone.fc = nn.Identity()
    backbone.eval()
    backbone.to(device)

    # Rebuild the head and load its trained weights
    head = nn.Linear(512, 10)
    head.load_state_dict(torch.load(weights_path, map_location=device))
    head.eval()
    head.to(device)

    return backbone, head

def predict_image(image_path, backbone, head, device="cpu"):
    transform = transforms.Compose([
        transforms.Grayscale(num_output_channels=3),
        transforms.Resize((128, 128)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    img = Image.open(image_path).convert("L")  # load as grayscale
    img_tensor = transform(img).unsqueeze(0).to(device)  # add batch dimension

    with torch.no_grad():
        features = backbone(img_tensor)
        outputs = head(features)
        probs = torch.softmax(outputs, dim=1)
        pred_idx = probs.argmax(dim=1).item()
        confidence = probs[0, pred_idx].item()

    return {"predicted_category": class_names[pred_idx], "confidence": confidence}

In [16]:
os.makedirs("data/sample_images", exist_ok=True)

# Load the ORIGINAL (un-normalized) test set just for exporting clean images
raw_test = datasets.FashionMNIST(root="./data", train=False, download=False)

# Pick 5 different images, ideally different categories
chosen_indices = [0, 1, 2, 3, 4]

for i in chosen_indices:
    img, label = raw_test[i]  # img is already a PIL Image here
    label_name = class_names[label].replace("/", "-")  # avoid slash in filename
    filename = f"data/sample_images/{i:02d}_{label_name}.png"
    img.save(filename)
    print("Saved:", filename)

Saved: data/sample_images/00_Ankle boot.png
Saved: data/sample_images/01_Pullover.png
Saved: data/sample_images/02_Trouser.png
Saved: data/sample_images/03_Trouser.png
Saved: data/sample_images/04_Shirt.png


In [18]:
backbone, head_loaded = load_classifier()
result = predict_image("data/sample_images/04_Shirt.png", backbone, head_loaded)
print(result)


{'predicted_category': 'Shirt', 'confidence': 0.46112632751464844}
